In [1]:
from pandas import DataFrame

In [ ]:
from streamlit_apps.apps.streamlit_app_research.application.services.asset_screening import AssetScreening10yPrice10yITRService, get_eligible_assets_10yPrice10yITR
df_1: DataFrame = get_eligible_assets_10yPrice10yITR(service=AssetScreening10yPrice10yITRService())

In [3]:
df_1.head(5)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro
0,PETR4,PETROBRAS,9512,26,15,4.883131e+08
1,BRAP4,BRADESPAR,18724,26,12,2.566157e+08
2,PETR3,PETROBRAS,9512,26,15,7.101182e+07
3,ITUB4,ITAUUNIBANCO,19348,25,15,4.710248e+07
4,B3SA3,B3,21610,18,15,3.337238e+07


In [ ]:
from data_providers.providers.yfinance_price_provider import YFinancePriceProvider
from streamlit_apps.apps.streamlit_app_research.application.services.asset_demonstration_service import AssetDemonstrationService

asset_demonstration_service = AssetDemonstrationService()

def _get_ret_stats(df: DataFrame) -> DataFrame:
    
    ret_stats = {}

    for row in df.itertuples():
        price_df = YFinancePriceProvider().get_asset_price(
            tickers=row.cod + ".SA", period="10y"
        )
        
        ret = price_df["Adj Close"].pct_change(1)
        
        try:
            divida_liquida = asset_demonstration_service.get_divida_liquida(row.codeCVM)["VL_CONTA_TRI"]
            media_divida_liquida = divida_liquida.mean()
            std_divida_liquida = divida_liquida.std()
            mediana_divida_liquida = divida_liquida.median()
        except:
            media_divida_liquida = None
            std_divida_liquida = None
            mediana_divida_liquida = None
            
        try:
            lucro_liquido = asset_demonstration_service.get_lucro_liquido(row.codeCVM)["VL_CONTA_TRI"]
            media_lucro_liquido = lucro_liquido.mean()
            std_lucro_liquido = lucro_liquido.std()
            mediana_lucro_liquido = lucro_liquido.median()
        except:
            media_lucro_liquido = None
            std_lucro_liquido = None
            mediana_lucro_liquido = None
            
        try:
            patrimonio_liquido = asset_demonstration_service.get_patrimonio_liquido(row.codeCVM)["VL_CONTA_TRI"]
            media_patrimonio_liquido = patrimonio_liquido.mean()
            std_patrimonio_liquido = patrimonio_liquido.std()
            mediana_patrimonio_liquido = patrimonio_liquido.median()
        except:
            media_patrimonio_liquido = None
            std_patrimonio_liquido = None
            mediana_patrimonio_liquido = None
        
        try:
            media_indice_de_alavancagem_financeira = (divida_liquida / patrimonio_liquido).mean()
            std_indice_de_alavancagem_financeira = (divida_liquida / patrimonio_liquido).std()
            mediana_indice_de_alavancagem_financeira = (divida_liquida / patrimonio_liquido).median()
        except:
            media_indice_de_alavancagem_financeira = None
            std_indice_de_alavancagem_financeira = None
            mediana_indice_de_alavancagem_financeira = None
                
        ret_stats[row.cod] = {
            "last_price": price_df["Close"].iloc[-1],
            "std_volume_financeiro": price_df["Volume"].std(),
            "media_ret": ret.mean(),
            "std_ret": ret.std(),
            "mediana_ret": ret.median(),
            "media_divida_liquida": media_divida_liquida,
            "std_divida_liquida": std_divida_liquida,
            "mediana_divida_liquida": mediana_divida_liquida,
            "media_lucro_liquido": media_lucro_liquido,
            "std_lucro_liquido": std_lucro_liquido,
            "mediana_lucro_liquido": mediana_lucro_liquido,
            "media_patrimonio_liquido": media_patrimonio_liquido,
            "std_patrimonio_liquido": std_patrimonio_liquido,
            "mediana_patrimonio_liquido": mediana_patrimonio_liquido,
            "media_indice_de_alavancagem_financeira": media_indice_de_alavancagem_financeira,
            "std_indice_de_alavancagem_financeira": std_indice_de_alavancagem_financeira,
            "mediana_indice_de_alavancagem_financeira": mediana_indice_de_alavancagem_financeira
            
        }
        
    return (
        DataFrame.from_dict(ret_stats, orient="index")
        .rename_axis("cod")
        .reset_index()
    )

df_2 = df_1.merge(_get_ret_stats(df_1), on="cod", how="left")

In [5]:
df_3 = df_2.dropna()
df_3.head(5)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro,last_price,std_volume_financeiro,media_ret,std_ret,...,mediana_divida_liquida,media_lucro_liquido,std_lucro_liquido,mediana_lucro_liquido,media_patrimonio_liquido,std_patrimonio_liquido,mediana_patrimonio_liquido,media_indice_de_alavancagem_financeira,std_indice_de_alavancagem_financeira,mediana_indice_de_alavancagem_financeira
0,PETR4,PETROBRAS,9512,26,15,4.883131e+08,48.919998,3.356366e+07,0.001423,0.025673,...,282080000.0,1.246664e+07,2.478704e+07,5785740.0,3.286764e+08,5.644192e+07,329838881.0,0.869465,0.354493,0.770174
1,BRAP4,BRADESPAR,18724,26,12,2.566157e+08,21.639999,3.191208e+06,0.001414,0.022856,...,593318.0,3.630645e+05,7.825377e+05,329133.0,9.164912e+06,1.498343e+06,9000800.0,0.029821,0.083887,0.067143
2,PETR3,PETROBRAS,9512,26,15,7.101182e+07,54.259998,1.040391e+07,0.001387,0.025874,...,282080000.0,1.246664e+07,2.478704e+07,5785740.0,3.286764e+08,5.644192e+07,329838881.0,0.869465,0.354493,0.770174
4,B3SA3,B3,21610,18,15,3.337238e+07,17.370001,2.073948e+07,0.000877,0.023441,...,5219348.0,7.372576e+05,6.963617e+05,606117.0,2.109343e+07,2.590245e+06,19673706.0,0.251455,0.218103,0.210317
5,COGN3,COGNA ON,17973,14,15,2.524634e+07,2.340000,2.532461e+07,-0.000097,0.032889,...,1096375.0,5.138671e+04,6.387283e+05,84778.0,1.092881e+07,5.237116e+06,12459688.0,0.324210,0.271504,0.254938


In [6]:
df_3.shape

(56, 23)

In [7]:
df_4 = df_3[df_3["media_lucro_liquido"] > 0].copy()
df_4 = df_4[df_4["last_price"] > 9]

In [8]:
df_4.shape

(45, 23)

In [9]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features = [
    # "ma_volume_financeiro",
    "std_volume_financeiro",
    # "media_ret",
    "std_ret",
    # "media_divida_liquida",
    # "std_divida_liquida",
    "mediana_divida_liquida",
    # "media_lucro_liquido",
    # "std_lucro_liquido",
    # "mediana_lucro_liquido",
    # "media_patrimonio_liquido",
    # "std_patrimonio_liquido",
    # "mediana_patrimonio_liquido",
    # "media_indice_de_alavancagem_financeira",
    # "std_indice_de_alavancagem_financeira",
    # "mediana_indice_de_alavancagem_financeira"
]

X = df_4[features].copy()

# log1p nas colunas com cauda pesada (volume e lucro têm ordens de magnitude de diferença)
# lucro pode ser negativo em alguns ativos -> usar signo * log1p(abs()) preserva o sinal
for col in [
    # "ma_volume_financeiro",
    "std_volume_financeiro",
    # "media_ret",
    # "std_ret",
    # "media_divida_liquida",
    # "std_divida_liquida",
    "mediana_divida_liquida",
    # "media_lucro_liquido",
    # "std_lucro_liquido",
    # "mediana_lucro_liquido",
    # "media_patrimonio_liquido",
    # "std_patrimonio_liquido",
    # "mediana_patrimonio_liquido",
    # "media_indice_de_alavancagem_financeira",
    # "std_indice_de_alavancagem_financeira",
    # "mediana_indice_de_alavancagem_financeira"
    
]:
    X[col] = np.sign(X[col]) * np.log1p(np.abs(X[col]))

X_scaled = StandardScaler().fit_transform(X)

# escolher k via silhouette (com 53 linhas, testar poucos k é suficiente)
scores = {}
k = 2
for k in range(k, 8):
    labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

best_k = max(scores, key=scores.get)
print("silhouette por k:", scores)
print("melhor k:", best_k)

kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)
df_4["cluster"] = kmeans.fit_predict(X_scaled)


silhouette por k: {2: 0.33143575779434287, 3: 0.407122093133319, 4: 0.4276346454841184, 5: 0.38541033286166476, 6: 0.3758550112464109, 7: 0.3455872614055485}
melhor k: 4


In [10]:
df_4.groupby("cluster")[features].mean()

,std_volume_financeiro,std_ret,mediana_divida_liquida
cluster,,,
0,5.431437e+06,0.030509,9.076529e+06
1,1.232942e+07,0.023077,5.443313e+07
2,4.707204e+06,0.018889,-7.565423e+06
3,2.605687e+06,0.019251,4.820155e+06


In [11]:
df_4["cluster"].value_counts()

cluster
3    18
1    13
0     9
2     5
Name: count, dtype: int64

In [12]:
df_4.tail(3)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro,last_price,std_volume_financeiro,media_ret,std_ret,...,media_lucro_liquido,std_lucro_liquido,mediana_lucro_liquido,media_patrimonio_liquido,std_patrimonio_liquido,mediana_patrimonio_liquido,media_indice_de_alavancagem_financeira,std_indice_de_alavancagem_financeira,mediana_indice_de_alavancagem_financeira,cluster
54,ISAE4,ISA ENERGIA,18376,26,15,1.136691e+06,27.280001,1.607786e+06,0.000625,0.013832,...,5.468268e+05,686936.667480,446861.5,1.096718e+07,5.419509e+06,11194258.0,0.365767,0.178109,0.357047,3
55,VIVT3,TELEF BRASIL,17671,26,15,1.123119e+06,30.510000,3.087759e+06,0.000619,0.016629,...,1.324869e+06,821024.833014,1211487.0,6.140671e+07,1.337005e+07,68768198.0,0.096686,0.062276,0.071236,3
56,AZZA3,AZZAS 2154,22349,15,15,8.575088e+05,15.770000,1.351243e+06,0.000275,0.026658,...,6.064298e+04,120365.577945,33311.5,1.642691e+06,2.042780e+06,684738.0,0.220832,0.158398,0.178344,3


In [13]:
# Interpretação dos clusters:
#
# Cluster 0:
# - Variabilidade do volume financeiro intermediária (≈ R$ 5,43 milhões).
# - Maior volatilidade dos retornos (≈ 3,05%).
# - Dívida líquida mediana positiva (≈ R$ 9,08 milhões).
# - Perfil de ativos mais volátil, com endividamento líquido moderado.
#
# Cluster 1:
# - Maior variabilidade do volume financeiro (≈ R$ 12,32 milhões).
# - Volatilidade dos retornos intermediária (≈ 2,31%).
# - Maior dívida líquida mediana (≈ R$ 54,43 milhões).
# - Perfil associado a maior escala financeira e maior endividamento líquido.
#
# Cluster 2:
# - Variabilidade do volume financeiro relativamente alta (≈ R$ 4,70 milhões).
# - Menor volatilidade dos retornos (≈ 1,89%).
# - Dívida líquida mediana negativa (≈ -R$ 7,57 milhões).
# - Perfil mais estável e com posição de caixa superior à dívida na mediana.
#
# Cluster 3:
# - Menor variabilidade do volume financeiro (≈ R$ 2,60 milhões).
# - Baixa volatilidade dos retornos (≈ 1,93%).
# - Dívida líquida mediana positiva, porém baixa (≈ R$ 4,82 milhões).
# - Perfil de menor atividade financeira e comportamento relativamente estável.
#
# Síntese:
# - Cluster 1: maior volume e maior endividamento.
# - Cluster 0: maior volatilidade dos retornos.
# - Cluster 2: baixa volatilidade e dívida líquida negativa.
# - Cluster 3: menor variabilidade do volume e baixa volatilidade.
#
# Observação:
# - Como volume financeiro e dívida líquida são medidas absolutas, parte da separação
#   pode estar relacionada ao porte das empresas.
# - Para uma interpretação econômica mais robusta, seria interessante utilizar também
#   métricas relativas, como dívida líquida/EBITDA, dívida líquida/patrimônio líquido
#   e margem líquida.

In [14]:
import plotly.express as px
from sklearn.decomposition import PCA

# TODO: Atualizar o número de componentes conforme a quantidade de features utilizadas no estudo.

pca = PCA(n_components=2, random_state=42)
components = pca.fit_transform(X_scaled)

df_4["pca_1"] = components[:, 0]
df_4["pca_2"] = components[:, 1]

ativo_destaque = "KLBN11"
ativo = df_4[df_4["cod"] == ativo_destaque]

var_explained = pca.explained_variance_ratio_

fig = px.scatter(
    df_4,
    x="pca_1",
    y="pca_2",
    color=df_4["cluster"].astype(str),
    hover_name="cod",
    hover_data={
        "cod": True,
        "asset": True,
        "media_ret": ":.4f",
        "std_ret": ":.4f",
        "ma_volume_financeiro": ":.2e",
        "std_volume_financeiro": ":.2e",
        "media_lucro_liquido": ":.2e",
        "std_lucro_liquido": ":.2e",
        "media_divida_liquida": ":.2e",
        "std_divida_liquida": ":.2e",
        
        "pca_1": False,
        "pca_2": False,
    },
    labels={
        "pca_1": f"PC1 ({var_explained[0]:.1%} var.)",
        "pca_2": f"PC2 ({var_explained[1]:.1%} var.)",
        "color": "Cluster",
    },
    title="Clusters de ativos — projeção PCA 2D",
    template="plotly_white",
    color_discrete_sequence=px.colors.qualitative.Set2,
)

fig.add_scatter(
    x=ativo["pca_1"],
    y=ativo["pca_2"],
    mode="markers+text",
    text=ativo["cod"],
    textposition="top center",
    marker=dict(
        size=14,
        color="red",
        symbol="diamond",
        line=dict(width=1, color="black"),
    ),
    name=f"Ativo: {ativo_destaque}",
)

fig.update_traces(
    marker=dict(size=12, line=dict(width=1, color="white")),
    selector=dict(mode="markers"),
)

fig.add_hline(y=0, line_dash="dash", line_color="gray", line_width=1)
fig.add_vline(x=0, line_dash="dash", line_color="gray", line_width=1)


fig.update_layout(
    legend_title_text="Cluster",
    hoverlabel=dict(bgcolor="white", font_size=13),
    height=550,
)

fig.show()

In [15]:
df_4.tail(3)

,cod,asset,codeCVM,anos_diferenca_preco,anos_diferenca_itr,ma_volume_financeiro,last_price,std_volume_financeiro,media_ret,std_ret,...,mediana_lucro_liquido,media_patrimonio_liquido,std_patrimonio_liquido,mediana_patrimonio_liquido,media_indice_de_alavancagem_financeira,std_indice_de_alavancagem_financeira,mediana_indice_de_alavancagem_financeira,cluster,pca_1,pca_2
54,ISAE4,ISA ENERGIA,18376,26,15,1.136691e+06,27.280001,1.607786e+06,0.000625,0.013832,...,446861.5,1.096718e+07,5.419509e+06,11194258.0,0.365767,0.178109,0.357047,3,-1.637113,1.243707
55,VIVT3,TELEF BRASIL,17671,26,15,1.123119e+06,30.510000,3.087759e+06,0.000619,0.016629,...,1211487.0,6.140671e+07,1.337005e+07,68768198.0,0.096686,0.062276,0.071236,3,-0.793382,0.793128
56,AZZA3,AZZAS 2154,22349,15,15,8.575088e+05,15.770000,1.351243e+06,0.000275,0.026658,...,33311.5,1.642691e+06,2.042780e+06,684738.0,0.220832,0.158398,0.178344,3,-0.415793,0.441952


In [16]:
# Avalia diferentes valores de k no KMeans para identificar uma quantidade
# adequada de clusters, considerando a separação dos grupos (silhouette)
# e a distribuição dos ativos entre eles.

import pandas as pd

for k in range(2, 8):

    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(X_scaled)

    silhouette = silhouette_score(X_scaled, labels)

    sizes = pd.Series(labels).value_counts().sort_index()

    print(
        f"k={k} | "
        f"silhouette={silhouette:.3f} | "
        f"tamanho={sizes.tolist()}"
    )

k=2 | silhouette=0.331 | tamanho=[23, 22]
k=3 | silhouette=0.407 | tamanho=[22, 5, 18]
k=4 | silhouette=0.428 | tamanho=[9, 13, 5, 18]
k=5 | silhouette=0.385 | tamanho=[8, 13, 11, 8, 5]
k=6 | silhouette=0.376 | tamanho=[8, 8, 4, 11, 1, 13]
k=7 | silhouette=0.346 | tamanho=[7, 8, 4, 12, 1, 8, 5]
